In [0]:
from cdp_dq_framework.models import EntityContract
from cdp_dq_framework.contract import run_contract_checks
from cdp_dq_framework.utils import read_csv, read_excel

In [0]:
tv = dbutils.jobs.taskValues
catalog = tv.get(taskKey="derive_config", key="catalog", debugValue=None)
schema_nm = tv.get(taskKey="derive_config", key="schema_nm", debugValue=None)
bucket = tv.get(taskKey="derive_config", key="bucket", debugValue=None)
file_key = tv.get(taskKey="derive_config", key="file_key", debugValue=None)
file_name = tv.get(taskKey="derive_config", key="file_name", debugValue=None)
app_id = tv.get(taskKey="derive_config", key="app_id", debugValue=None)
entity_id = tv.get(taskKey="derive_config", key="entity_id", debugValue=None)
excn_id = tv.get(taskKey="derive_config", key="excn_id", debugValue=None)
service_type = tv.get(taskKey="derive_config", key="service_type", debugValue=None)
pre_contract_file_path = tv.get(taskKey="derive_config", key="pre_contract_file_path", debugValue=None)
file_ext = file_name.split(".")[-1]

In [0]:
row = spark.sql(f"SELECT * FROM {catalog}.{schema_nm}.dc_entity_mstr WHERE app_id='{app_id}' AND entity_id='{entity_id}'").limit(1).collect()[0].asDict()
contract = EntityContract.from_row(row)

In [0]:
# read — metadata-driven (port of setparam): delimiter/header/encoding/quote handled in utils
read_purpose = "contract_check"
if file_ext in ("csv", "txt", "flat", "dat", "tab"):
    file_data_df = read_csv(spark, pre_contract_file_path, contract, read_purpose, service_type)
elif file_ext in ("xlsx", "xls"):
    file_data_df = read_excel(spark, pre_contract_file_path, contract, read_purpose, service_type)
else:
    raise Exception(f"Invalid file received: {file_name}")

In [0]:
file_data_df.show(5)

In [0]:
result = run_contract_checks(file_data_df, contract, file_name=file_name)  # <-- reusable checks

In [0]:
result.passed

In [0]:
result.checks

In [0]:
result.failed_checks

In [0]:
tv = dbutils.jobs.taskValues
task_values = {
    "result_status": result.passed,
    "should_proceed": True if result.passed in ("SUCCESS", "BYPASSED") else False,
    "result": dict(result.__dict__) if result is not None else None,
    "file_ext": file_ext
}
for key, val in task_values.items():
    tv.set(key=key, value=val)